# Trial {N} — <hypothesis in one line>

**Key insight:** <TBD — fill at the end (also goes into trials.json)>

> ▶ = run the cell · ✍️ = write here before continuing · ⛔ never "Run All" — full rules: `instructions.md`

## 1. ✍️ Read and summarize

Read `context.md` (physics), `trial_00.ipynb` (experiment anchor), **all previous trial
notebooks**, and `trials.json`. Then write below **your current understanding of the
situation**: where the campaign stands, what the last trials showed, what this trial should test.

<agent writes here>

In [ ]:
# 2. ▶ Load history + propose 10 candidates (with EI scores)
import json, glob, re, os, sys

NOTEBOOK_DIR = os.getcwd()                    # this experiment folder (pagho, trials.json)
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, 'src')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)    # climb to repo root (dir containing src/)
sys.path.insert(0, NOTEBOOK_DIR)              # for pagho
sys.path.insert(0, REPO_ROOT)                 # for src/
os.chdir(REPO_ROOT)                           # data/... resolve from repo root
TRIALS_PATH = os.path.join(NOTEBOOK_DIR, 'trials.json')

# --- structured history ---
trials = json.load(open(TRIALS_PATH))['trials']
best = min((t for t in trials if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print(f"{len(trials)} trials so far | current best: {best['trial_id'] if best else None} "
      f"(obj={best['objective']:.4f})" if best else f"{len(trials)} trials so far | no best yet")

# --- memory index (Key insight of every previous trial; trial_00 first) ---
for p in sorted(glob.glob(os.path.join(NOTEBOOK_DIR, 'trial_*.ipynb'))):
    nb2 = json.load(open(p))
    ki = '?'
    for c in nb2['cells']:
        if c['cell_type'] == 'markdown':
            m = re.search(r'\*\*Key insight:\*\*\s*(.*)', ''.join(c['source']))
            if m: ki = m.group(1).strip()
            break
    tag = '  <-- READ FIRST (experiment anchor)' if os.path.basename(p) == 'trial_00.ipynb' else ''
    print(f'  {os.path.basename(p)}: {ki}{tag}')

# --- propose 10 candidates ---
from pagho import propose, DEFAULT_SPACE
candidates = propose(trials, space=DEFAULT_SPACE, n_candidates=10, seed=None)
print()
print(f"{'ID':>2} | {'EI':>6} | params")
print('-' * 78)
for i, cand in enumerate(candidates, 1):
    cfg = '  '.join(f'{k}={v}' for k, v in cand['config'].items())
    print(f'{i:2d} | {cand["score"]:.4f} | {cfg}')

## 3. ✍️ Analyze and choose

Analyze the 10 candidates using **physics reasoning** (context.md failure modes) and the
notebook history. Note any **disagreement with the EI ranking**. **Choose ONE** and justify
in detail: which failure mode it attacks, what you expect to happen, what would confirm or
refute it.

<agent writes here>

In [ ]:
# 4. ▶ Set your choice and run the trial  ⛔ ~1h — do not interrupt unless obviously broken
INDEX = 1              # <-- your chosen candidate (1-based, from the table in cell 2)
TRIAL_ID = 'trial_XXX' # <-- next free number

assert 1 <= INDEX <= len(candidates), 'bad INDEX'
CHOSEN = candidates[INDEX - 1]['config']
print('chosen:', CHOSEN)

import traceback, time
from pagho import run_trial, compute_objective
t0 = time.time()
try:
    results = run_trial(CHOSEN)
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
    objective, uncertainty = compute_objective(results)
    print(f'objective (MSE vs true) = {objective:.4f} ± {uncertainty:.4f}')
    # TODO: detailed per-experiment results table + plots (convergence, distributions)
except Exception:
    traceback.print_exc()
    results, objective, uncertainty = None, None, None

## 5. ✍️ Analyze the results

Write a comprehensive analysis: what happened vs expectations, **hypothesis confirmed or
refuted** (evidence, not vibes), what was learned about the physics and the hyperparameters,
**ideas worth keeping**, and the **next hypothesis**.

<agent writes here>

In [ ]:
# 6. ▶ Update trials.json — fill the two strings below first, then run
SUMMARY = '<one-line summary of this trial, from your analysis above>'
KEY_INSIGHT = '<one sentence — same as the title cell>'

entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': objective,
    'uncertainty': uncertainty,
    'summary': SUMMARY,
    'key_insight': KEY_INSIGHT,
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)

## 7. ↺ Loop back

Return to `instructions.md` and start the next trial (copy this template to the next number).

If this trial motivates a **structural change** (score, likelihood, model, protocol):
describe it here and **request Anuar's permission** — do NOT run it yourself.